In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

import logging
logging.basicConfig(level=logging.INFO)

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Literal
import torch
import pandas
from pandas import DataFrame
from torch import Tensor
from tsdm.encoders import time2float
from torch.utils.data import SequentialSampler, Dataset, DataLoader

rng = np.random.default_rng()
np.set_printoptions(4)

In [3]:
from tsdm.datasets import ETTh1

ds = ETTh1.dataset
target = "OT"
forecasting_horizon: Literal[24, 48, 168, 336, 960] = (24,)
observation_horizon: Literal[24, 48, 96, 168, 336, 720] = (96,)
test_metric: Literal["MSE", "MAE"] = ("MSE",)

train_dataset = ds[:"2017-06-30"]  # inclusive range!
valid_dataset = ds["2017-07-01":"2017-10-31"]  # inclusive range!
trial_dataset = ds["2017-11-01":"2018-02-28"]  # inclusive range!
trial_dataset_copy = trial_dataset.copy()

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2017-11-01 00:00:00,11.989,4.622,9.594,2.736,2.315,1.066,9.567
2017-11-01 01:00:00,11.989,4.019,9.417,2.416,2.284,1.188,9.708
2017-11-01 02:00:00,10.650,3.550,8.244,1.919,2.193,1.249,9.426
2017-11-01 03:00:00,10.516,3.550,8.564,2.168,2.071,1.127,9.426
2017-11-01 04:00:00,10.516,2.947,7.889,1.599,2.254,1.157,9.567
...,...,...,...,...,...,...,...
2018-02-28 19:00:00,10.449,1.005,7.214,0.036,3.320,0.548,5.768
2018-02-28 20:00:00,10.449,0.938,7.320,-0.036,3.229,0.579,5.276
2018-02-28 21:00:00,9.578,0.067,6.858,-0.817,2.711,0.548,4.995


## Preprocessing

In [4]:
from sklearn.preprocessing import StandardScaler

encoder = StandardScaler()
encoder.fit(train_dataset)
display(
    DataFrame.from_dict(
        {"mean": encoder.mean_, "stdv": encoder.scale_},
        orient="index",
        columns=train_dataset.columns,
    )
)

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
mean,7.909795,2.019931,5.045573,0.763926,2.787863,0.771786,17.169089
stdv,5.820164,2.080180,5.534651,1.922766,1.020136,0.642108,9.122404


In [5]:
encoder.transform(trial_dataset, copy=False)
trial_dataset

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2017-11-01 00:00:00,0.700875,1.250887,0.821809,1.025644,-0.463529,0.458201,-0.833343
2017-11-01 01:00:00,0.700875,0.961008,0.789829,0.859217,-0.493918,0.648200,-0.817886
2017-11-01 02:00:00,0.470812,0.735547,0.577891,0.600736,-0.583121,0.743199,-0.848799
2017-11-01 03:00:00,0.447789,0.735547,0.635709,0.730237,-0.702713,0.553200,-0.848799
2017-11-01 04:00:00,0.447789,0.445668,0.513750,0.434309,-0.523325,0.599921,-0.833343
...,...,...,...,...,...,...,...
2018-02-28 19:00:00,0.436277,-0.487905,0.391791,-0.378583,0.521634,-0.348517,-1.249790
2018-02-28 20:00:00,0.436277,-0.520114,0.410943,-0.416029,0.432430,-0.300239,-1.303723
2018-02-28 21:00:00,0.286625,-0.938828,0.327469,-0.822215,-0.075346,-0.348517,-1.334526


In [6]:
encoder.inverse_transform(trial_dataset, copy=False)
pandas.testing.assert_frame_equal(trial_dataset, trial_dataset_copy)

In [7]:
splits["train"]

NameError: name 'splits' is not defined

### Data Loading

In [ ]:
from tsdm.datasets import SequenceDataset
from tsdm.util.samplers import SequenceSampler

In [ ]:
time_encoder = time2float
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

train_dataset.drop(columns="OT")

_T = time_encoder(train_dataset.index)
_X = train_dataset.drop(columns="OT").values
_Y = train_dataset["OT"].values

T = torch.tensor(_T, device=device, dtype=dtype)
X = torch.tensor(_X, device=device, dtype=dtype)
Y = torch.tensor(_Y, device=device, dtype=dtype)

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, tensors: list[Tensor]):
        assert all(len(x) == len(tensors[0]) for x in tensors)
        self.tensors = tensors

    def __len__(self):
        return len(self.tensors[0])

    def __getitem__(self, idx):
        return [x[idx] for x in self.tensors]


class SequenceSampler(torch.utils.data.Sampler):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __iter__(self):
        print(len(self.data), self.seq_len)
        for idx in range(len(self.data) - self.seq_len):
            yield range(idx, idx + self.seq_len)

In [ ]:
train_dataset = SequenceDataset([T, X, Y])
sampler = SequenceSampler(train_dataset, 2)
samples = list(iter(DataLoader(train_dataset, shuffle=True)));

In [ ]:
first, second, last = (samples[0], samples[1], samples[-1])
first, second, last

# Implemented Task

In [ ]:
from tsdm.tasks import ETDatasetInformer

In [ ]:
from tsdm.datasets import DATASETS

In [ ]:
task = ETDatasetInformer("ETTh2")

In [ ]:
dloader = task.get_dataloader("test")

In [ ]:
task.splits["test"].values.mean(axis=0)

In [ ]:
for item in dloader:
    t, x, y = item
torch.mean(x, dim=(0, 1)), torch.std(x, dim=(0, 1))

In [ ]:
task.dataset.dataset